In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
# For Jupyter Notebook: keep 23_delivery_performance.csv in the same folder.
# For Google Colab, uncomment the two lines below to upload the CSV.

# from google.colab import files
# uploaded = files.upload()

file_name = "23_delivery_performance.csv"
df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
display(df.head())

In [ ]:
print("Number of rows and columns:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())
print("\\nData types:")
print(df.dtypes)
print("\\nDataset information:")
df.info()

In [ ]:
print("Missing values in each column:")
print(df.isnull().sum())

if "Shipment_ID" in df.columns:
    print("\\nDuplicate Shipment_ID records:", df["Shipment_ID"].duplicated().sum())
else:
    print("Shipment_ID column was not found. Check the column names in your CSV.")

In [ ]:
# Missing Delivery_Date is kept as missing because inventing a delivery date
# would give an incorrect delivery duration. Such records are excluded from
# duration calculations later.
if "Delivery_Date" in df.columns:
    df["Delivery_Date"] = df["Delivery_Date"].replace(r"^\\s*$", np.nan, regex=True)

# Fill missing Shipping_Cost with the median (robust to extreme values).
if "Shipping_Cost" in df.columns:
    df["Shipping_Cost"] = pd.to_numeric(df["Shipping_Cost"], errors="coerce")
    median_cost = df["Shipping_Cost"].median()
    df["Shipping_Cost"] = df["Shipping_Cost"].fillna(median_cost)
    print("Missing Shipping_Cost values filled with median:", median_cost)

print("Missing values after handling:")
print(df.isnull().sum())

In [ ]:
for col in ["Order_Date", "Delivery_Date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print(df[["Order_Date", "Delivery_Date"]].dtypes)

In [ ]:
df["Delivery_Days"] = (df["Delivery_Date"] - df["Order_Date"]).dt.days
print(df[["Order_Date", "Delivery_Date", "Delivery_Days"]].head())
print("Records with unavailable/invalid delivery duration:", df["Delivery_Days"].isna().sum())

In [ ]:
total_shipments = len(df)

# Treat a status containing the word 'deliver' as delivered.
# This also matches statuses such as 'Delivered' and 'Delivered Late'.
if "Delivery_Status" in df.columns:
    delivered_mask = df["Delivery_Status"].astype(str).str.contains("deliver", case=False, na=False)
    delivered_shipments = int(delivered_mask.sum())
else:
    delivered_mask = pd.Series(False, index=df.index)
    delivered_shipments = 0
    print("Delivery_Status column not found; delivered count cannot be calculated.")

success_percentage = (delivered_shipments / total_shipments * 100) if total_shipments else 0

print("Total shipments:", total_shipments)
print("Delivered shipments:", delivered_shipments)
print(f"Overall delivery success percentage: {success_percentage:.2f}%")

In [ ]:
if "Shipping_Method" in df.columns:
    avg_by_method = df.groupby("Shipping_Method")["Delivery_Days"].mean().sort_values()
    display(avg_by_method.to_frame("Average_Delivery_Days"))
else:
    print("Shipping_Method column not found.")

In [ ]:
if "Delivery_Partner" in df.columns:
    partner_ranking = df.groupby("Delivery_Partner").agg(
        Shipment_Count=("Shipment_ID", "count") if "Shipment_ID" in df.columns else ("Delivery_Partner", "size"),
        Average_Delivery_Days=("Delivery_Days", "mean")
    ).sort_values(["Shipment_Count", "Average_Delivery_Days"], ascending=[False, True])
    display(partner_ranking)
else:
    print("Delivery_Partner column not found.")

In [ ]:
if "Shipping_Method" in df.columns and "Delivery_Status" in df.columns:
    success_data = df.copy()
    success_data["Is_Delivered"] = delivered_mask.astype(int)
    method_success = success_data.groupby("Shipping_Method")["Is_Delivered"].mean().mul(100).round(2)
    display(method_success.to_frame("Delivery_Success_Percentage"))

    method_success.plot(kind="bar", figsize=(8, 4), ylabel="Success rate (%)",
                        title="Delivery success rate by shipping method")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Shipping_Method or Delivery_Status column not found.")

In [ ]:
if "Shipping_Method" in df.columns and "Delivery_Status" in df.columns:
    success_data = df.copy()
    success_data["Is_Delivered"] = delivered_mask.astype(int)
    method_success = success_data.groupby("Shipping_Method")["Is_Delivered"].mean().mul(100).round(2)
    display(method_success.to_frame("Delivery_Success_Percentage"))

    method_success.plot(kind="bar", figsize=(8, 4), ylabel="Success rate (%)",
                        title="Delivery success rate by shipping method")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Shipping_Method or Delivery_Status column not found.")

In [ ]:
longest_deliveries = df.dropna(subset=["Delivery_Days"]).nlargest(10, "Delivery_Days")
show_cols = [c for c in ["Shipment_ID", "Delivery_Partner", "Shipping_Method",
                         "Order_Date", "Delivery_Date", "Delivery_Days"] if c in df.columns]
display(longest_deliveries[show_cols])

In [ ]:
if "Delivery_Status" in df.columns:
    delivery_counts = df["Delivery_Status"].value_counts(dropna=False).to_dict()
else:
    delivery_counts = {}

if "Delivery_Partner" in df.columns:
    unique_partners = set(df["Delivery_Partner"].dropna().unique())
else:
    unique_partners = set()

print("Delivery counts by status (dictionary):")
print(delivery_counts)
print("\\nUnique delivery partners (set):")
print(unique_partners)

In [ ]:
# Editable example day limits. Update these if your teacher gave other limits.
ON_TIME_MAX_DAYS = 3
SLIGHTLY_DELAYED_MAX_DAYS = 7

def label_shipment(days):
    if pd.isna(days):
        return "Unknown"
    elif days <= ON_TIME_MAX_DAYS:
        return "On-Time"
    elif days <= SLIGHTLY_DELAYED_MAX_DAYS:
        return "Slightly Delayed"
    else:
        return "Highly Delayed"

df["Delivery_Label"] = df["Delivery_Days"].apply(label_shipment)
display(df[["Delivery_Days", "Delivery_Label"]].head(10))
print(df["Delivery_Label"].value_counts())

In [ ]:
valid_days = df["Delivery_Days"].dropna().to_numpy()

if len(valid_days) > 0:
    print("Mean delivery days:", np.mean(valid_days))
    print("Median delivery days:", np.median(valid_days))
    print("Standard deviation:", np.std(valid_days))
else:
    print("No valid delivery durations available.")

In [ ]:
valid_df = df.dropna(subset=["Delivery_Days"]).copy()

if len(valid_df) > 0:
    Q1 = valid_df["Delivery_Days"].quantile(0.25)
    Q3 = valid_df["Delivery_Days"].quantile(0.75)
    IQR = Q3 - Q1
    upper_limit = Q3 + 1.5 * IQR

    outliers = valid_df[valid_df["Delivery_Days"] > upper_limit]
    print("Q1:", Q1, " Q3:", Q3, " IQR:", IQR)
    print("Upper outlier limit:", upper_limit)
    print("Unusually long deliveries:", len(outliers))
    display(outliers)

    if "Order_Date" in valid_df.columns:
        valid_df["Order_Month"] = valid_df["Order_Date"].dt.to_period("M").astype(str)
        monthly_performance = valid_df.groupby("Order_Month")["Delivery_Days"].agg(
            Shipment_Count="count", Average_Delivery_Days="mean"
        )
        display(monthly_performance)

        monthly_performance["Average_Delivery_Days"].plot(
            kind="line", marker="o", figsize=(9, 4),
            title="Average delivery duration by month", ylabel="Average days"
        )
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
else:
    print("No valid delivery durations available for IQR analysis.")

In [ ]:
class Shipment:
    def __init__(self, shipment_id, partner, delivery_days):
        self.shipment_id = shipment_id
        self.partner = partner
        self.delivery_days = delivery_days

    def display_details(self):
        print("Shipment ID:", self.shipment_id)
        print("Delivery Partner:", self.partner)
        print("Delivery Days:", self.delivery_days)

# Create an object using the first available record.
if len(df) > 0:
    first = df.iloc[0]
    shipment = Shipment(
        first.get("Shipment_ID", "Not available"),
        first.get("Delivery_Partner", "Not available"),
        first.get("Delivery_Days", np.nan)
    )
    shipment.display_details()
else:
    print("Dataset is empty.")

In [ ]:
recommendations = [
    "1. Identify shipping methods with the lowest delivery success rate and investigate the reasons.",
    "2. Review delivery partners with high shipment volumes and long average delivery durations.",
    "3. Monitor shipments that cross the IQR upper limit and investigate their causes.",
    "4. Improve dispatch planning and route allocation during months with longer average delivery times.",
    "5. Track delivery status regularly and inform customers early when a shipment is delayed."
]

for recommendation in recommendations:
    print(recommendation)

In [ ]:
df.to_csv("delivery_analysis_cleaned.csv", index=False)
print("Cleaned dataset saved as delivery_analysis_cleaned.csv")